# 02 — Baseline Model Training
**4 Models | EGFR | Accuracy + Resource Tracking**

This notebook trains all 4 candidate models on the same EGFR dataset and logs:
- Accuracy: R², RMSE, Spearman ρ
- Resources: Peak VRAM, training time, CO2 (CodeCarbon)

Each model is trained independently and results saved to `./results/`.

> **Prerequisite:** `01_data.ipynb` must have run — `./data/train.csv` must exist.

## 0. Imports & Config

In [ ]:
import sys, os, json, time, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

from utils.metrics import compute_metrics, ResourceTracker, print_metrics
from utils.mol_utils import batch_smiles_to_morgan, smiles_to_graph

with open('./config.json') as f:
    CONFIG = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CONFIG['use_amp'] and torch.cuda.is_available()
SEED = CONFIG['random_seed']
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device:  {DEVICE}')
print(f'AMP:     {USE_AMP}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Load data
df_train = pd.read_csv('./data/train.csv')
df_val   = pd.read_csv('./data/val.csv')
df_test  = pd.read_csv('./data/test.csv')

print(f'\nTrain: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

## MODEL 1 — Random Forest + Morgan Fingerprints
Classical QSAR baseline. Runs on CPU — no GPU needed.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from codecarbon import EmissionsTracker
import joblib

print('=' * 55)
print('MODEL 1: Random Forest + Morgan Fingerprints')
print('=' * 55)

# Featurize
print('Computing Morgan fingerprints (radius=2, 2048 bits)...')
X_train, vi = batch_smiles_to_morgan(df_train['Drug'].tolist())
y_train = df_train['Y_log'].iloc[vi].values

X_val, vi_v = batch_smiles_to_morgan(df_val['Drug'].tolist())
y_val = df_val['Y_log'].iloc[vi_v].values

X_test, vi_t = batch_smiles_to_morgan(df_test['Drug'].tolist())
y_test = df_test['Y_log'].iloc[vi_t].values

print(f'  X_train shape: {X_train.shape}')

# Track resources
res_tracker = ResourceTracker('rf_morgan', './results/')
co2_tracker = EmissionsTracker(project_name='rf_morgan', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_rf.csv')

res_tracker.start()
co2_tracker.start()

# Train
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train, y_train)

kg_co2 = co2_tracker.stop()
resource_stats = res_tracker.stop()

# Evaluate
val_pred  = rf.predict(X_val)
test_pred = rf.predict(X_test)

val_metrics  = compute_metrics(y_val,  val_pred,  prefix='val_')
test_metrics = compute_metrics(y_test, test_pred, prefix='test_')
all_metrics  = {**val_metrics, **test_metrics, 'kg_co2': round(kg_co2, 6)}

# Save
joblib.dump(rf, './models/rf_morgan.pkl')
with open('./results/metrics_rf_morgan.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('\nValidation metrics:')
print_metrics(val_metrics)
print('\nTest metrics:')
print_metrics(test_metrics)
print(f'\nCO2: {kg_co2*1000:.4f} g | Model saved to models/rf_morgan.pkl')

## MODEL 2 — AttentiveFP (Graph Attention Network)
Lightweight GNN — designed specifically for molecular property prediction.

In [ ]:
import deepchem as dc
from deepchem.models import AttentiveFPModel
from codecarbon import EmissionsTracker

print('=' * 55)
print('MODEL 2: AttentiveFP')
print('=' * 55)

# DeepChem featurizer for graph models
featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)

print('Featurizing molecules as graphs...')
X_train_g = featurizer.featurize(df_train['Drug'].tolist())
X_val_g   = featurizer.featurize(df_val['Drug'].tolist())
X_test_g  = featurizer.featurize(df_test['Drug'].tolist())

# Build DeepChem datasets
ds_train = dc.data.NumpyDataset(X=X_train_g, y=df_train['Y_log'].values.reshape(-1,1))
ds_val   = dc.data.NumpyDataset(X=X_val_g,   y=df_val['Y_log'].values.reshape(-1,1))
ds_test  = dc.data.NumpyDataset(X=X_test_g,  y=df_test['Y_log'].values.reshape(-1,1))

# Clear GPU before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()

res_tracker = ResourceTracker('attentivefp', './results/')
co2_tracker = EmissionsTracker(project_name='attentivefp', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_attentivefp.csv')

res_tracker.start()
co2_tracker.start()

model_afp = AttentiveFPModel(
    mode='regression',
    n_tasks=1,
    num_layers=3,
    num_timesteps=2,
    graph_feat_size=200,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=CONFIG['batch_size'],
    model_dir='./models/attentivefp/'
)

# Train with validation
best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(50):
    loss = model_afp.fit(ds_train, nb_epoch=1)
    val_scores = model_afp.evaluate(ds_val, [dc.metrics.Metric(dc.metrics.rms_score)])
    val_rmse = val_scores['rms_score']

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:>3} | loss={loss:.4f} | val_rmse={val_rmse:.4f}')

    # Early stopping
    if val_rmse < best_val_loss:
        best_val_loss = val_rmse
        patience_counter = 0
        model_afp.save_checkpoint()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

kg_co2 = co2_tracker.stop()
resource_stats = res_tracker.stop()

# Evaluate
model_afp.restore()
val_pred  = model_afp.predict(ds_val).flatten()
test_pred = model_afp.predict(ds_test).flatten()

val_metrics  = compute_metrics(ds_val.y.flatten(),  val_pred,  prefix='val_')
test_metrics = compute_metrics(ds_test.y.flatten(), test_pred, prefix='test_')
all_metrics  = {**val_metrics, **test_metrics, 'kg_co2': round(kg_co2, 6)}

with open('./results/metrics_attentivefp.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('\nValidation metrics:')
print_metrics(val_metrics)
print('\nTest metrics:')
print_metrics(test_metrics)
print(f'\nCO2: {kg_co2*1000:.4f} g')

torch.cuda.empty_cache()

## MODEL 3 — MPNN (Message Passing Neural Network via DeepChem)

In [ ]:
from deepchem.models import MPNNModel
from codecarbon import EmissionsTracker

print('=' * 55)
print('MODEL 3: MPNN (Message Passing Neural Network)')
print('=' * 55)

# MPNN uses different featurizer
mpnn_featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
X_train_m = mpnn_featurizer.featurize(df_train['Drug'].tolist())
X_val_m   = mpnn_featurizer.featurize(df_val['Drug'].tolist())
X_test_m  = mpnn_featurizer.featurize(df_test['Drug'].tolist())

ds_train_m = dc.data.NumpyDataset(X=X_train_m, y=df_train['Y_log'].values.reshape(-1,1))
ds_val_m   = dc.data.NumpyDataset(X=X_val_m,   y=df_val['Y_log'].values.reshape(-1,1))
ds_test_m  = dc.data.NumpyDataset(X=X_test_m,  y=df_test['Y_log'].values.reshape(-1,1))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

res_tracker = ResourceTracker('mpnn', './results/')
co2_tracker = EmissionsTracker(project_name='mpnn', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_mpnn.csv')

res_tracker.start()
co2_tracker.start()

model_mpnn = MPNNModel(
    n_tasks=1,
    mode='regression',
    node_out_feats=64,
    edge_hidden_feats=32,
    num_step_message_passing=3,
    num_step_set2set=3,
    num_layer_set2set=2,
    learning_rate=1e-3,
    batch_size=CONFIG['batch_size'],
    model_dir='./models/mpnn/'
)

best_val = float('inf')
patience_counter = 0

for epoch in range(50):
    loss = model_mpnn.fit(ds_train_m, nb_epoch=1)
    val_scores = model_mpnn.evaluate(ds_val_m, [dc.metrics.Metric(dc.metrics.rms_score)])
    val_rmse = val_scores['rms_score']

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:>3} | loss={loss:.4f} | val_rmse={val_rmse:.4f}')

    if val_rmse < best_val:
        best_val = val_rmse
        patience_counter = 0
        model_mpnn.save_checkpoint()
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'  Early stopping at epoch {epoch}')
            break

kg_co2 = co2_tracker.stop()
resource_stats = res_tracker.stop()

model_mpnn.restore()
val_pred  = model_mpnn.predict(ds_val_m).flatten()
test_pred = model_mpnn.predict(ds_test_m).flatten()

val_metrics  = compute_metrics(ds_val_m.y.flatten(),  val_pred,  prefix='val_')
test_metrics = compute_metrics(ds_test_m.y.flatten(), test_pred, prefix='test_')
all_metrics  = {**val_metrics, **test_metrics, 'kg_co2': round(kg_co2, 6)}

with open('./results/metrics_mpnn.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('\nValidation metrics:')
print_metrics(val_metrics)
print('\nTest metrics:')
print_metrics(test_metrics)
print(f'\nCO2: {kg_co2*1000:.4f} g')

torch.cuda.empty_cache()

## MODEL 4 — ChemBERTa-2 (Pretrained Transformer)
Heaviest model — but pretrained means fewer epochs needed.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from codecarbon import EmissionsTracker

print('=' * 55)
print('MODEL 4: ChemBERTa-2')
print('=' * 55)

MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'

# ── Dataset class ───────────────────────────────────────────────
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            smiles_list, truncation=True,
            padding='max_length', max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label':          self.labels[idx]
        }

# ── Model class ─────────────────────────────────────────────────
class ChemBERTaRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token
        return self.regressor(cls).squeeze(-1)

# Load tokenizer + build datasets
print('Loading ChemBERTa tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

BATCH = 32  # Smaller batch for transformer on 8GB
train_ds = SMILESDataset(df_train['Drug'].tolist(), df_train['Y_log'].values, tokenizer)
val_ds   = SMILESDataset(df_val['Drug'].tolist(),   df_val['Y_log'].values,   tokenizer)
test_ds  = SMILESDataset(df_test['Drug'].tolist(),  df_test['Y_log'].values,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

res_tracker = ResourceTracker('chemberta2', './results/')
co2_tracker = EmissionsTracker(project_name='chemberta2', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_chemberta2.csv')

res_tracker.start()
co2_tracker.start()

model_cb = ChemBERTaRegressor(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model_cb.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler    = GradScaler(enabled=USE_AMP)
criterion = nn.MSELoss()

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(20):
    model_cb.train()
    train_losses = []
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)

        with autocast(enabled=USE_AMP):          # Mixed precision
            preds = model_cb(ids, mask)
            loss  = criterion(preds, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_cb.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    scheduler.step()

    # Validation
    model_cb.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            with autocast(enabled=USE_AMP):
                preds = model_cb(batch['input_ids'].to(DEVICE),
                                  batch['attention_mask'].to(DEVICE))
            val_preds.extend(preds.cpu().float().numpy())
            val_true.extend(batch['label'].numpy())

    val_rmse = np.sqrt(np.mean((np.array(val_preds) - np.array(val_true))**2))
    avg_train = np.mean(train_losses)
    print(f'  Epoch {epoch+1:>2} | train_loss={avg_train:.4f} | val_rmse={val_rmse:.4f}')

    if val_rmse < best_val_loss:
        best_val_loss = val_rmse
        patience_counter = 0
        torch.save(model_cb.state_dict(), './models/chemberta2_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= 4:
            print(f'  Early stopping at epoch {epoch+1}')
            break

kg_co2 = co2_tracker.stop()
resource_stats = res_tracker.stop()

# Final evaluation with best checkpoint
model_cb.load_state_dict(torch.load('./models/chemberta2_best.pt'))
model_cb.eval()

def predict_loader(loader):
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            with autocast(enabled=USE_AMP):
                p = model_cb(batch['input_ids'].to(DEVICE),
                              batch['attention_mask'].to(DEVICE))
            preds.extend(p.cpu().float().numpy())
            trues.extend(batch['label'].numpy())
    return np.array(preds), np.array(trues)

val_pred,  val_true  = predict_loader(val_loader)
test_pred, test_true = predict_loader(test_loader)

val_metrics  = compute_metrics(val_true,  val_pred,  prefix='val_')
test_metrics = compute_metrics(test_true, test_pred, prefix='test_')
all_metrics  = {**val_metrics, **test_metrics, 'kg_co2': round(kg_co2, 6)}

with open('./results/metrics_chemberta2.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print('\nValidation metrics:')
print_metrics(val_metrics)
print('\nTest metrics:')
print_metrics(test_metrics)
print(f'\nCO2: {kg_co2*1000:.4f} g')

torch.cuda.empty_cache()

## Summary — All 4 Models

In [ ]:
from utils.metrics import load_all_results, efficiency_score
import pandas as pd

results = load_all_results('./results/')

rows = []
for name, data in results.items():
    rows.append({
        'Model':         name,
        'Val R²':        data.get('val_r2', '-'),
        'Val RMSE':      data.get('val_rmse', '-'),
        'Spearman ρ':    data.get('val_spearman_rho', '-'),
        'VRAM (MB)':     data.get('peak_vram_mb', 0),
        'Train (min)':   data.get('train_time_min', '-'),
        'CO2 (mg)':      round(data.get('kg_co2', 0) * 1e6, 2),
        'Efficiency':    data.get('efficiency_score', '-'),
    })

df_summary = pd.DataFrame(rows)
print('\nAll Models — Summary')
print(df_summary.to_string(index=False))
df_summary.to_csv('./results/baseline_summary.csv', index=False)
print('\nSaved to results/baseline_summary.csv')
print('\n✓ Ready for 03_comparison.ipynb')